In [179]:
import numpy as np
import matplotlib.pyplot as plt

In [180]:
def triangular_lattice_pbc(N, Lx, Ly, seed=None):
    """
    Place N particles on a triangular lattice compatible with PBCs
    in a box [0, Lx] x [0, Ly].

    Returns positions array of shape (N, 2).
    """
    # Step 1: Find best factorization N = nx * ny
    # Target ratio: ny/nx = 2*Ly / (sqrt(3)*Lx)
    target_ratio = 2.0 * Ly / (np.sqrt(3) * Lx)
    
    best = None
    best_err = np.inf
    for nx in range(1, N + 1):
        if N % nx == 0:
            ny = N // nx
            err = abs(ny / nx - target_ratio)
            if err < best_err:
                best_err = err
                best = (nx, ny)
    
    nx, ny = best
    print(f"Using nx={nx}, ny={ny}, ratio={ny/nx:.4f} (target {target_ratio:.4f})")
    print(f"Lattice strain: {best_err/target_ratio*100:.2f}%")

    # Step 2: Primitive vectors fitted to box
    a1 = np.array([Lx / nx, 0.0])
    a2 = np.array([Lx / (2 * nx), Ly / ny])
    
    # Step 3: Generate all lattice sites
    positions = []
    for j in range(ny):
        for i in range(nx):
            r = i * a1 + j * a2
            # Wrap into box (should be clean, but just in case)
            r[0] = r[0] % Lx
            r[1] = r[1] % Ly
            positions.append(r)

    positions = np.array(positions)
    
    # Optional: add small random perturbation to break symmetry
    if seed is not None:
        rng = np.random.default_rng(seed)
        eps = 0.025 * Lx / nx          # 5% of lattice spacing
        positions += rng.uniform(-eps, eps, positions.shape)
        positions[:, 0] %= Lx
        positions[:, 1] %= Ly

    return positions, a1, a2

In [205]:
# --- Example: reproduce paper parameters ---
L = 36.0
nsk = 0.16
N = int(nsk * L**2)   # = 207, round as needed

pos, a1, a2 = triangular_lattice_pbc(N+1, L, L)

# Verify density
print(f"N = {len(pos)}, nsk = {len(pos)/L**2:.4f}")

Using nx=13, ny=16, ratio=1.2308 (target 1.1547)
Lattice strain: 6.59%
N = 208, nsk = 0.1605


In [206]:
np.sqrt(3)/2

np.float64(0.8660254037844386)

In [207]:
def structure_factor(pos, Lx, Ly, kmag_max=15):
    """Compute S(k) on a grid and return the 2D array."""
    N = len(pos)
    # Reciprocal lattice grid
    dkx = 2 * np.pi / Lx
    dky = 2 * np.pi / Ly
    nk = int(kmag_max / min(dkx, dky))
    
    kx = np.arange(-nk, nk + 1) * dkx
    ky = np.arange(-nk, nk + 1) * dky
    KX, KY = np.meshgrid(kx, ky)
    
    # S(k) = |sum_j exp(i k . r_j)|^2 / N
    phase = KX[:, :, None] * pos[:, 0] + KY[:, :, None] * pos[:, 1]
    Sk = np.abs(np.sum(np.exp(1j * phase), axis=-1))**2 / N
    
    return KX, KY, Sk

In [211]:
Lx = 36.
Ly = 36.
eta = 0.16
N = int(eta * Lx * Ly)

pos, a1, a2 = triangular_lattice_pbc(N, Lx, Ly)
sf = structure_factor(pos, Lx, Ly)
# Verify density
print(f"N = {len(pos)}, nsk = {len(pos)/Lx/Ly:.4f}")

Using nx=23, ny=9, ratio=0.3913 (target 1.1547)
Lattice strain: 66.11%
N = 207, nsk = 0.1597


In [214]:
16*16

256

In [212]:
%matplotlib qt
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(pos[:, 0], pos[:, 1], 'o', markersize=4)
ax.set_aspect(1)

In [213]:
%matplotlib qt
import matplotlib.colors as mcolors
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sf[2], extent=(sf[0].min(), sf[0].max(), sf[1].min(), sf[1].max()),
origin='lower',
cmap='gray', norm=mcolors.LogNorm(vmin=sf[2][sf[2] > 0].min()))
ax.set_xlabel('kx')
ax.set_ylabel('ky')
ax.set_title('Structure Factor S(k) [log scale]')
fig.colorbar(im, ax=ax)

In [204]:
0.2*(36**2)

259.2